In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark=SparkSession.builder \
                  .appName('silver_layer') \
                  .master('local[*]') \
                  .config('spark.sql.shuffle.partitions', '8') \
                  .config('spark.driver.memory', '2g') \
                  .getOrCreate()

users_dataframe

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
schema_of_users=StructType([StructField('user_id',IntegerType(),True),
                            StructField('first_name',StringType(),True),
                            StructField('last_name',StringType(),True),
                            StructField('email',StringType(),True),
                            StructField('age',IntegerType(),True),
                            StructField('gender',StringType(),True),
                            StructField('state',StringType(),True),
                            StructField('street_address',StringType(),True),
                            StructField('postal_code',StringType(),True),
                            StructField('city',StringType(),True),
                            StructField('country',StringType(),True),
                            StructField('latitude',DoubleType(),True),
                            StructField('longitude',DoubleType(),True),
                            StructField('traffic_source',StringType(),True),
                            StructField('registered_at',TimestampType(),True)
                           ])

In [ ]:
users_df = spark.read.csv("hdfs:///Data/Bronze/Raw_Looker_Ecommerce_BigQuery_Dataset/users.csv",
                          header=True,
                          schema=schema_of_users,
                          nullValue='null')

transformations_of_users_dataframe

In [ ]:
from pyspark.sql.functions import col, trim, regexp_replace, when, initcap, upper, lower

users_df = users_df.drop('postal_code', 'latitude', 'longitude')

users_df = users_df.fillna({"city": "Unknown"})

users_df = users_df.withColumn(
        'first_name',
            initcap(
                trim(
                    regexp_replace(
                        regexp_replace(
                            regexp_replace(trim(col('first_name')), r"\s+", " "),
                            r"[^\p{L}\s'\-]", ""
                        ),
                        r"\s+", " "
                    )
                )
            )
        )

users_df = users_df.withColumn(
        'last_name',
            initcap(
                trim(
                    regexp_replace(
                        regexp_replace(
                            regexp_replace(trim(col('last_name')), r"\s+", " "),
                            r"[^\p{L}\s'\-]", ""
                        ),
                        r"\s+", " "
                    )
                )
            )
        )

users_df = users_df.withColumn(
        'email',
            lower(
                regexp_replace(trim(col('email')), r"\s+", ""))
            )

users_df = users_df.withColumn(
        'gender',
            upper(regexp_replace(trim(col('gender')), r"\s+", ""))
        )

users_df = users_df.withColumn(
        'state',
            initcap(
                trim(
                    regexp_replace(
                        regexp_replace(trim(col('state')), r"\s+", " "),
                        r"[^\p{L}\s'\-]", ""
                    )
                )
            )
        )

users_df = users_df.withColumn(
        'street_address',
            regexp_replace(trim(col('street_address')), r"\s+", " ")
        )

users_df = users_df.withColumn(
        'city',
            initcap(
                trim(
                    regexp_replace(
                        regexp_replace(trim(col('city')), r"\s+", " "),
                        r"[^\p{L}\s'\-]", ""
                    )
                )
            )
        )

users_df = users_df.withColumn(
        'country',
            initcap(regexp_replace(trim(col('country')), r"\s+", " "))
        )

users_df = users_df.withColumn(
        'traffic_source',
            initcap(regexp_replace(trim(col('traffic_source')), r"\s+", " "))
        )

users_df = users_df.dropDuplicates(["user_id"])

products_dataframe

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
schema_of_products=StructType([StructField('product_id',IntegerType(),True),
                            StructField('cost',DoubleType(),True),
                            StructField('category',StringType(),True),
                            StructField('product_name',StringType(),True),
                            StructField('brand',StringType(),True),
                            StructField('retail_price',DoubleType(),True),
                            StructField('department',StringType(),True),
                            StructField('sku',StringType(),True),
                            StructField('distribution_center_id',IntegerType(),True),
                           ])

In [ ]:
products_df = spark.read.csv("hdfs:///Data/Bronze/Raw_Looker_Ecommerce_BigQuery_Dataset/products.csv",
                          header=True,
                          schema=schema_of_products,
                          nullValue='null')

transformations_of_products_dataframe

In [ ]:
from pyspark.sql.functions import col, trim, regexp_replace, initcap, when
from pyspark.sql.types import DecimalType

products_df = products_df.drop('sku')

products_df = products_df.fillna({"product_name": "Unknown", "brand": "Unknown"})

products_df = products_df.withColumn("cost", col("cost").cast(DecimalType(10, 2)))

products_df = products_df.withColumn(
        'category',
            initcap(regexp_replace(trim(col('category')), r"\s+", " "))
        )

products_df = products_df.withColumn(
        'product_name',
            regexp_replace(trim(col('product_name')), r"\s+", " ")
        )

products_df = products_df.withColumn(
        'brand',
            regexp_replace(trim(col('brand')), r"\s+", " ")
        )

products_df = products_df.withColumn("retail_price", col("retail_price").cast(DecimalType(10, 2)))

products_df = products_df.withColumn(
        'department',
            initcap(regexp_replace(trim(col('department')), r"\s+", " "))
        )

products_df = products_df.dropDuplicates(["product_id"])

distribution_centers_dataframe

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
schema_of_distribution_centers=StructType([StructField('distribution_center_id',IntegerType(),True),
                            StructField('distribution_center_name',StringType(),True),
                            StructField('latitude',DoubleType(),True),
                            StructField('longitude',DoubleType(),True),
                           ])

In [ ]:
distribution_centers_df = spark.read.csv("hdfs:///Data/Bronze/Raw_Looker_Ecommerce_BigQuery_Dataset/distribution_centers.csv",
                          header=True,
                          schema=schema_of_distribution_centers,
                          nullValue='null')

transformations_of_distribution_centers_dataframe

In [ ]:
from pyspark.sql.functions import col, trim, regexp_extract, regexp_replace, initcap, upper

distribution_centers_df = distribution_centers_df.drop('latitude', 'longitude')

distribution_centers_df = distribution_centers_df.withColumn(
    "distribution_center_state",
    regexp_replace(
        regexp_extract(upper(trim(col("distribution_center_name"))), r"\s([A-Z]{2}(?:/[A-Z]{2})?)$", 1),
        "/", "&"
    )
)

distribution_centers_df = distribution_centers_df.withColumn(
    "distribution_center_name",
    trim(regexp_replace(col("distribution_center_name"), r"\s+[A-Za-z]{2}(/[A-Za-z]{2})?$", ""))
)

cleaned_name = regexp_replace(
    regexp_replace(trim(col("distribution_center_name")), r"\s+", " "),
    r"[^\p{L}\s'\-/]", ""
)

distribution_centers_df = distribution_centers_df.withColumn(
    "distribution_center_name",
    regexp_replace(initcap(regexp_replace(cleaned_name, "/", "/ ")), "/ ", "/")
)

distribution_centers_df = distribution_centers_df.dropDuplicates(["distribution_center_id"])

order_items_dataframe

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
schema_of_order_items=StructType([StructField('order_item_id',IntegerType(),True),
                            StructField('order_id',IntegerType(),True),
                            StructField('user_id',IntegerType(),True),
                            StructField('product_id',IntegerType(),True),
                            StructField('inventory_item_id',IntegerType(),True),
                            StructField('status_name',StringType(),True),
                            StructField('created_at',TimestampType(),True),
                            StructField('shipped_at',TimestampType(),True),
                            StructField('delivered_at',TimestampType(),True),
                            StructField('returned_at',TimestampType(),True),
                            StructField('sale_price',DoubleType(),True)
                           ])

In [ ]:
order_items_df = spark.read.csv("hdfs:///Data/Bronze/Raw_Looker_Ecommerce_BigQuery_Dataset/order_items.csv",
                          header=True,
                          schema=schema_of_order_items,
                          nullValue='null')

transformations_of_order_items_dataframe

In [ ]:
from pyspark.sql.functions import col, trim, regexp_replace, initcap, create_map, lit, when
from pyspark.sql.types import DecimalType

order_items_df = order_items_df.drop('inventory_item_id')

order_items_df = order_items_df.withColumn(
        'status_name',
        initcap(
            regexp_replace(trim(col('status_name')), r"\s+", " "))
        )

status_mapping = {
    "Processing": 0,
    "Shipped": 1,
    "Complete": 2,
    "Returned": 3,
    "Cancelled": 4
}

mapping_expr = create_map( *[item for kv in status_mapping.items() for item in (lit(kv[0]), lit(kv[1]))])

order_items_df = order_items_df.withColumn(
        "status_id",
        when(col("status_name").isNull(), lit(None).cast("int"))
        .otherwise(mapping_expr[trim(col("status_name"))])
    )

order_items_df = order_items_df.withColumn(
        "status_id",
        when(col("status_id").isNull(), lit(-1)).otherwise(col("status_id"))
    )

order_items_df = order_items_df.withColumn('sale_price', col('sale_price').cast(DecimalType(10, 2)))

order_items_df = order_items_df.dropDuplicates(["order_item_id"])

events_dataframe

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, DoubleType
schema_of_events=StructType([StructField('event_id',IntegerType(),True),
                            StructField('user_id',DoubleType(),True),
                            StructField('sequence_number',IntegerType(),True),
                            StructField('session_id',StringType(),True),
                            StructField('created_at',TimestampType(),True),
                            StructField('ip_address',StringType(),True),
                            StructField('city',StringType(),True),
                            StructField('state',StringType(),True),
                            StructField('postal_code',StringType(),True),
                            StructField('browser',StringType(),True),
                            StructField('traffic_source',StringType(),True),
                            StructField('uri',StringType(),True),
                            StructField('event_type',StringType(),True)
                           ])

In [ ]:
events_df = spark.read.csv("hdfs:///Data/Bronze/Raw_Looker_Ecommerce_BigQuery_Dataset/events.csv",
                          header=True,
                          schema=schema_of_events,
                          nullValue='null')

transformations_of_events_dataframe

In [ ]:
from pyspark.sql.functions import col, trim, regexp_replace, when, initcap, upper, lower,length

events_df = events_df.drop('ip_address', 'postal_code')

events_df   = events_df.fillna({"city": "Unknown"})

events_df = events_df.withColumn("user_id", col("user_id").cast("int"))

events_df = events_df.withColumn('session_id', regexp_replace(trim(col('session_id')), r"\s+", ""))

events_df = events_df.withColumn('uri', regexp_replace(trim(col('uri')), r"\s+", ""))

events_df = events_df.withColumn(
        'city',
            initcap(
                trim(
                    regexp_replace(
                        regexp_replace(trim(col('city')), r"\s+", " "),
                        r"[^\p{L}\s'\-]", ""
                    )
                )
            )
        )

events_df = events_df.withColumn(
        'state',
            initcap(
                trim(
                    regexp_replace(
                        regexp_replace(trim(col('state')), r"\s+", " "),
                        r"[^\p{L}\s'\-]", ""
                    )
                )
            )
        )

events_df = events_df.withColumn(
    'browser',
    when(length(trim(col('browser'))) == 2, upper(trim(col('browser'))))
    .otherwise(initcap(regexp_replace(trim(col('browser')), r"\s+", " ")))
)

events_df = events_df.withColumn(
        'traffic_source',
            initcap(regexp_replace(trim(col('traffic_source')), r"\s+", " "))
        )

events_df = events_df.withColumn(
        'event_type',
            initcap(regexp_replace(trim(col('event_type')), r"\s+", " "))
        )

events_df = events_df.dropDuplicates(["event_id"])

invalid records for future investigations

In [ ]:
silver_quarantine = "hdfs:///Data/Silver/Invalid_Looker_Ecommerce_BigQuery_Dataset"

def _delete_path(path):
    """Delete a stale HDFS path; does nothing if it doesn't exist"""
    p = spark._jvm.org.apache.hadoop.fs.Path(path)
    fs = p.getFileSystem(spark._jsc.hadoopConfiguration())
    if fs.exists(p):
        fs.delete(p, True)

def split_invalid(df, name, condition):
    """
    condition = boolean expression for VALID rows.
    Invalid rows (NOT (condition is True)) are written to quarantine.
    Returns the cleaned df.
    """
    df_cached = df.cache()

    invalid = df_cached.filter(~condition | condition.isNull())
    n_invalid = invalid.count()

    _delete_path(f"{silver_quarantine}/{name}")

    if n_invalid > 0:
        invalid.coalesce(1).write.mode("overwrite").parquet(f"{silver_quarantine}/{name}")
        print(f"{name}: quarantined {n_invalid} invalid rows")
    else:
        print(f"{name}: no invalid rows")

    cleaned = df_cached.filter(condition).cache()
    cleaned.count()
    df_cached.unpersist()
    return cleaned

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql import functions as F

users_df = split_invalid(
    users_df, "users",
    (col("user_id").isNotNull()) &
    (col("age").between(0, 100)) &
    (col("gender").isin("M", "F")) &
    (col("registered_at").isNotNull()) &
    (col("registered_at") <= F.current_timestamp())
)

products_df = split_invalid(
    products_df, "products",
    (col("product_id").isNotNull()) &
    (col("cost") > 0) & (col("cost") < 100000) &
    (col("retail_price") > 0) & (col("retail_price") < 100000) &
    (col("distribution_center_id").isNotNull())
)

distribution_centers_df = split_invalid(
    distribution_centers_df, "distribution_centers",
    col("distribution_center_id").isNotNull()
)

order_items_df = split_invalid(
    order_items_df, "order_items",
    (col("order_item_id").isNotNull()) &
    (col("order_id").isNotNull()) &
    (col("user_id").isNotNull()) &
    (col("product_id").isNotNull()) &
    (col("sale_price") > 0) & (col("sale_price") < 100000) &
    (col("status_id").isin(0, 1, 2, 3, 4, -1)) &
    (col("created_at").isNotNull()) &
    (col("shipped_at").isNull()   | (col("shipped_at")   >= col("created_at"))) &
    (col("delivered_at").isNull() | (col("delivered_at") >= col("shipped_at"))) &
    (col("returned_at").isNull()  | (col("returned_at")  >= col("delivered_at")))
)

events_df = split_invalid(
    events_df, "events",
    (col("event_id").isNotNull()) &
    (col("sequence_number") >= 1) &
    (col("created_at").isNotNull()) &
    (col("created_at") <= F.current_timestamp())
)

load_data_to_hdfs

In [ ]:
silver = "hdfs:///Data/Silver/Cleaned_Looker_Ecommerce_BigQuery_Dataset"

users_df.coalesce(1).write.mode("overwrite").parquet(f"{silver}/users")
products_df.coalesce(1).write.mode("overwrite").parquet(f"{silver}/products")
distribution_centers_df.coalesce(1).write.mode("overwrite").parquet(f"{silver}/distribution_centers")
order_items_df.coalesce(1).write.mode("overwrite").parquet(f"{silver}/order_items")
events_df.coalesce(4).write.mode("overwrite").parquet(f"{silver}/events")